In [ ]:
# 1. Install Library to handle Kaggle Downloads
!pip install opendatasets h5py torch torchvision pandas -q

import opendatasets as od
import os

# 2. Download the 'Eye Gaze' dataset (Contains processed MPIIGaze)
# This is a 2GB dataset. It might take 1-2 minutes.
print("⬇️ Downloading real MPIIGaze dataset from Kaggle...")
od.download("https://www.kaggle.com/datasets/4quant/eye-gaze")

# 3. Verify the file exists
if os.path.exists("eye-gaze/real_gaze.h5"):
    print("✅ SUCCESS: Found real_gaze.h5 (Real MPIIGaze Data)")
else:
    print("❌ ERROR: Download failed. Please check your Kaggle credentials.")

⬇️ Downloading real MPIIGaze dataset from Kaggle...
Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: priyanshupraneet
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/4quant/eye-gaze


100%|██████████| 4.46G/4.46G [00:42<00:00, 112MB/s]



✅ SUCCESS: Found real_gaze.h5 (Real MPIIGaze Data)


In [ ]:
import h5py
import numpy as np

# Open the H5 file
file_path = "eye-gaze/real_gaze.h5"

try:
    with h5py.File(file_path, 'r') as f:
        print("📁 H5 File Keys:", list(f.keys()))

        # We expect keys like 'images' and 'gaze' or 'labels'
        # Let's verify the first item shape
        first_key = list(f.keys())[0]
        print(f"   Shape of '{first_key}':", f[first_key].shape)

except Exception as e:
    print(f"Error reading file: {e}")

📁 H5 File Keys: ['image']
Error reading file: 'Group' object has no attribute 'shape'


In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
import os
import scipy.io as sio

class MPIIGazeMatDataset(Dataset):
    def __init__(self, root_dir, transform=None, limit_participants=5):
        self.transform = transform
        self.images = []
        self.labels = []

        print(f"📂 Scanning '{root_dir}' for .mat files...")

        participants = sorted(os.listdir(root_dir))[:limit_participants]

        for p_id in participants:
            p_path = os.path.join(root_dir, p_id)
            if not os.path.isdir(p_path): continue

            mat_files = [f for f in os.listdir(p_path) if f.endswith('.mat')]

            for mat_file in mat_files:
                full_path = os.path.join(p_path, mat_file)
                try:
                    # 1. Load Matlab File
                    mat = sio.loadmat(full_path, struct_as_record=False, squeeze_me=True)

                    if 'data' not in mat: continue

                    data = mat['data']
                    for side in ['left', 'right']:
                        if hasattr(data, side):
                            eye_data = getattr(data, side)

                            # 2. Extract Data
                            imgs = eye_data.image
                            gazes = eye_data.gaze

                            # 3. Fix Image Shapes (N, H, W) -> (N, H, W)
                            # Ensure images are (N, H, W)
                            if len(imgs.shape) == 3 and imgs.shape[0] < imgs.shape[2]:
                                imgs = np.transpose(imgs, (2, 0, 1))

                            # 4. CRITICAL FIX: Robust Loop
                            # We iterate one by one and check if the gaze is valid
                            if len(imgs) == len(gazes):
                                for i in range(len(imgs)):
                                    g = gazes[i]

                                    # CHECK: Is 'g' a valid vector of 3 numbers?
                                    # The previous error happened because 'g' was sometimes a single number (scalar)
                                    if isinstance(g, np.ndarray) and g.shape == (3,):
                                        self.images.append(imgs[i])
                                        self.labels.append(g)
                                    else:
                                        # Silently skip bad data points
                                        continue

                except Exception as e:
                    continue # Skip bad files entirely

        self.length = len(self.images)
        print(f"✅ Successfully loaded {self.length} CLEAN image/gaze pairs.")

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # 1. Image
        img_arr = self.images[idx]
        if img_arr.dtype != np.uint8:
            img_arr = (img_arr).astype(np.uint8)
        img = Image.fromarray(img_arr).convert('RGB')

        # 2. Label
        gaze_vec = self.labels[idx]

        # Convert 3D Vector -> Pitch/Yaw
        pitch = np.arcsin(-gaze_vec[1])
        yaw = np.arctan2(-gaze_vec[0], -gaze_vec[2])

        label = torch.tensor([pitch, yaw], dtype=torch.float32)

        if self.transform:
            img = self.transform(img)

        return img, label

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import models, transforms
import time
from google.colab import files

# --- 1. CONFIGURATION ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NORMALIZED_DIR = "eye-gaze/normalized" # Path to your .mat files
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 5

print(f"🚀 Setting up training on {DEVICE}...")

# --- 2. PREPARE DATA ---
# Standard ResNet preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Initialize the Robust Dataset
# limit_participants=5 keeps it fast and RAM-safe.
# You can increase to 10 if you have High-RAM Colab.
dataset = MPIIGazeMatDataset(NORMALIZED_DIR, transform=transform, limit_participants=5)

if len(dataset) == 0:
    raise ValueError("❌ No images loaded! Check if 'eye-gaze/normalized' path is correct.")

# Split 80/20
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"✅ Data Ready: {len(train_dataset)} Training samples, {len(val_dataset)} Validation samples.")

# --- 3. SETUP MODEL ---
print("🚀 Loading ResNet18...")
model = models.resnet18(weights='DEFAULT')

# Modify last layer for Regression (2 outputs: Pitch, Yaw)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)

model = model.to(DEVICE)

# Loss and Optimizer
criterion = nn.MSELoss() # Mean Squared Error is best for regression
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# --- 4. TRAINING LOOP ---
print("🚀 Starting Training Loop...")
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if i % 100 == 0:
            print(f"   [Epoch {epoch+1}] Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

    # Validation Phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    print(f"✅ Epoch [{epoch+1}/{EPOCHS}] Completed | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

total_time = (time.time() - start_time) / 60
print(f"\n🎉 Training Finished in {total_time:.1f} minutes!")

# --- 5. SAVE & DOWNLOAD ---
torch.save(model.state_dict(), "resnet18.pt")
print("💾 Model saved as 'resnet18.pt'")

try:
    files.download('resnet18.pt')
except Exception as e:
    print(f"⚠️ Auto-download failed: {e}")
    print("👉 Click the folder icon on the left, find 'resnet18.pt', click the 3 dots -> Download.")

🚀 Setting up training on cuda...
📂 Scanning 'eye-gaze/normalized' for .mat files...
✅ Successfully loaded 267816 CLEAN image/gaze pairs.
✅ Data Ready: 214252 Training samples, 53564 Validation samples.
🚀 Loading ResNet18...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 170MB/s]


🚀 Starting Training Loop...
   [Epoch 1] Batch 0/6696 | Loss: 0.2380
   [Epoch 1] Batch 100/6696 | Loss: 0.0055
   [Epoch 1] Batch 200/6696 | Loss: 0.0060
   [Epoch 1] Batch 300/6696 | Loss: 0.0029
   [Epoch 1] Batch 400/6696 | Loss: 0.0048
   [Epoch 1] Batch 500/6696 | Loss: 0.0041
   [Epoch 1] Batch 600/6696 | Loss: 0.0048
   [Epoch 1] Batch 700/6696 | Loss: 0.0022
   [Epoch 1] Batch 800/6696 | Loss: 0.0027
   [Epoch 1] Batch 900/6696 | Loss: 0.0040
   [Epoch 1] Batch 1000/6696 | Loss: 0.0017
   [Epoch 1] Batch 1100/6696 | Loss: 0.0087
   [Epoch 1] Batch 1200/6696 | Loss: 0.0039
   [Epoch 1] Batch 1300/6696 | Loss: 0.0023
   [Epoch 1] Batch 1400/6696 | Loss: 0.0015
   [Epoch 1] Batch 1500/6696 | Loss: 0.0027
   [Epoch 1] Batch 1600/6696 | Loss: 0.0029
   [Epoch 1] Batch 1700/6696 | Loss: 0.0020
   [Epoch 1] Batch 1800/6696 | Loss: 0.0041
   [Epoch 1] Batch 1900/6696 | Loss: 0.0048
   [Epoch 1] Batch 2000/6696 | Loss: 0.0021
   [Epoch 1] Batch 2100/6696 | Loss: 0.0041
   [Epoch 1] Bat

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>